# Làm sạch dữ liệu kết quả sinh viên

Notebook này kiểm tra dữ liệu, gọi quy trình làm sạch dùng chung và xuất dữ liệu đã xử lý. Giá trị thiếu của **đặc trưng** được giữ lại để pipeline điền sau khi chia train/test, tránh rò rỉ dữ liệu.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().resolve()
if project_root.name in {'src', 'notebooks'}:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import CLEAN_DATA_PATH, DATA_PATH, TARGET_COLUMN
from src.data_cleaning import clean_student_data, save_clean_data

## 1. Đọc và khảo sát dữ liệu gốc

In [ ]:
raw_data = pd.read_csv(DATA_PATH)
print('Kích thước:', raw_data.shape)
print('Số dòng trùng:', int(raw_data.duplicated().sum()))
print('Tổng ô thiếu:', int(raw_data.isna().sum().sum()))
display(raw_data.head())

In [ ]:
missing_report = pd.DataFrame({
    'Missing_Count': raw_data.isna().sum(),
    'Missing_Percent': (raw_data.isna().mean() * 100).round(2),
}).query('Missing_Count > 0').sort_values('Missing_Count', ascending=False)
display(missing_report)

In [ ]:
if not missing_report.empty:
    missing_report['Missing_Percent'].plot(kind='bar', figsize=(10, 4))
    plt.title('Tỷ lệ dữ liệu thiếu theo thuộc tính')
    plt.ylabel('Tỷ lệ thiếu (%)')
    plt.xlabel('Thuộc tính')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()

## 2. Làm sạch bằng quy trình dùng chung

In [ ]:
cleaned_data, cleaning_summary = clean_student_data(raw_data)
pd.Series(cleaning_summary.to_dict(), name='Giá trị').to_frame()

Các ô thiếu của đặc trưng **không được điền tại đây**. Median và most-frequent imputation nằm trong `Pipeline`, được fit chỉ trên tập huấn luyện sau khi chia dữ liệu.

In [ ]:
numeric_columns = cleaned_data.select_dtypes(include='number').columns
outlier_rows = []
for column in numeric_columns:
    values = cleaned_data[column].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_rows.append({
        'Column': column,
        'Lower_Bound': lower,
        'Upper_Bound': upper,
        'Outlier_Count': int(((values < lower) | (values > upper)).sum()),
    })
outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)

## 3. Kiểm tra và xuất dữ liệu sạch

In [ ]:
assert cleaned_data.columns.tolist() == raw_data.columns.tolist()
assert cleaned_data.duplicated().sum() == 0
assert cleaned_data[TARGET_COLUMN].notna().all()
assert cleaned_data[TARGET_COLUMN].between(0, 100).all()
assert cleaned_data[TARGET_COLUMN].max() == 100

output_path = save_clean_data(cleaned_data, CLEAN_DATA_PATH)
print('Đã lưu:', output_path.relative_to(project_root))
print('Kích thước dữ liệu sạch:', cleaned_data.shape)
print('Tổng ô thiếu giữ lại cho pipeline:', int(cleaned_data.drop(columns=[TARGET_COLUMN]).isna().sum().sum()))

In [ ]:
saved_data = pd.read_csv(CLEAN_DATA_PATH)
changed_cells = int((
    raw_data.fillna('__NA__').astype(str)
    != saved_data.fillna('__NA__').astype(str)
).sum().sum())

print('Số ô thay đổi so với dữ liệu gốc:', changed_cells)
print('Điểm lớn nhất sau làm sạch:', saved_data[TARGET_COLUMN].max())
display(saved_data[saved_data[TARGET_COLUMN] == 100].head())